step 1: Load the dataset(NLTK movie reviews: 1000 positive + 1000 negative

In [16]:
import nltk
nltk.download('movie_reviews')
from nltk.corpus import movie_reviews

texts  = [movie_reviews.raw(fid) for fid in movie_reviews.fileids()]
labels = [1 if movie_reviews.categories(fid)[0] == 'pos' else 0 for fid in movie_reviews.fileids()]
print(len(texts), 'reviews | positive:', sum(labels), '| negative:', len(labels) - sum(labels))

2000 reviews | positive: 1000 | negative: 1000


[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


step 2: Train / Test split

In [17]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels)
print(len(X_train), 'train |', len(X_test), 'test')

1600 train | 400 test


step 3: Build and train the model

In [18]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

model = make_pipeline(
    TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2),
    LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

Pipeline(steps=[('tfidfvectorizer',
                 TfidfVectorizer(min_df=2, ngram_range=(1, 2),
                                 stop_words='english')),
                ('logisticregression', LogisticRegression(max_iter=1000))])

step 4: Evaluate

In [19]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
preds = model.predict(X_test)
print('Accuracy:', round(accuracy_score(y_test, preds), 3))
print(classification_report(y_test, preds, target_names=['negative', 'positive']))
print('Confusion matrix:\n', confusion_matrix(y_test, preds))

Accuracy: 0.812
              precision    recall  f1-score   support

    negative       0.83      0.79      0.81       200
    positive       0.80      0.84      0.82       200

    accuracy                           0.81       400
   macro avg       0.81      0.81      0.81       400
weighted avg       0.81      0.81      0.81       400

Confusion matrix:
 [[157  43]
 [ 32 168]]


step 5: Five manual test cases

In [20]:
tests = [
    ("An absolutely brilliant film, I loved every minute of it.",   "positive"),
    ("Boring, predictable and way too long. A total waste of time.", "negative"),
    ("The acting was great but the story made no sense at all.",     "negative"),  # mixed
    ("Not bad at all, I was pleasantly surprised.",                  "positive"),  # negation
    ("Oh great, another sequel nobody asked for. Just wonderful.",   "negative"),  # sarcasm
]
probs = model.predict_proba([t for t, _ in tests])[:, 1]
for (text, truth), p in zip(tests, probs):
    pred = 'positive' if p >= 0.5 else 'negative'
    print(f"{'OK  ' if pred == truth else 'FAIL'} | true={truth:8} pred={pred:8} P(pos)={p:.2f} | {text}")

OK   | true=positive pred=positive P(pos)=0.53 | An absolutely brilliant film, I loved every minute of it.
OK   | true=negative pred=negative P(pos)=0.30 | Boring, predictable and way too long. A total waste of time.
FAIL | true=negative pred=positive P(pos)=0.57 | The acting was great but the story made no sense at all.
FAIL | true=positive pred=negative P(pos)=0.41 | Not bad at all, I was pleasantly surprised.
FAIL | true=negative pred=positive P(pos)=0.51 | Oh great, another sequel nobody asked for. Just wonderful.


In [21]:
import numpy as np
vec, clf = model.steps[0][1], model.steps[1][1]
names = np.array(vec.get_feature_names_out())
order = np.argsort(clf.coef_[0])
print('Most NEGATIVE words:', list(names[order[:10]]))
print('Most POSITIVE words:', list(names[order[-10:]]))

Most NEGATIVE words: ['bad', 'movie', 'worst', 'plot', 'boring', 'supposed', 'script', 'harry', 'stupid', 'waste']
Most POSITIVE words: ['mulan', 'perfect', 'excellent', 'world', 'truman', 'best', 'family', 'war', 'life', 'great']


step 6: What the model learned

In [22]:
import joblib
joblib.dump(model, 'sentiment_model.joblib')
print('saved sentiment_model.joblib')

saved sentiment_model.joblib


## 8. Reflection: where does the model fail, and why?

**Overall performance:** The model scored **81.2% accuracy** on 400 unseen reviews. Precision and recall were balanced at about 0.80 to 0.84 for both classes. It made 43 false positives (negative reviews read as positive) and 32 false negatives. It is clearly learning something, but nearly 1 in 5 predictions is wrong.

**Manual tests: 2 of 5 correct.** It got the two simple cases right ("brilliant film" and "boring, waste of time"). It failed all three tricky ones:

| Test | Truth | Prediction | P(pos) | Why it failed |
|---|---|---|---|---|
| "Acting was great but the story made no sense" | negative | positive | 0.57 | Mixed sentiment: "great" pulls positive and the model can't weigh the "but" clause |
| "Not bad at all, I was pleasantly surprised" | positive | negative | 0.41 | Negation: "bad" is one of the strongest negative words, and the model ignores "not" |
| "Oh great, another sequel nobody asked for. Just wonderful." | negative | positive | 0.51 | Sarcasm: positive words ("great", "wonderful") with a negative meaning |

**Why it fails:** TF-IDF with Logistic Regression is a bag-of-words model. It assigns each word a weight and adds them up, with no understanding of word order, negation, or tone. Even "not bad" is only a bigram feature it rarely saw in training.

**Other things the model learned:**
- Its top positive words include `mulan`, `truman`, `world`, `war` and `family`, and its top negative words include `harry`, `movie` and `plot`. These are not sentiment words. They are movie titles and topics that happened to appear in the training reviews, so the model partly memorised the dataset rather than learning sentiment. This is overfitting to dataset quirks.
- Several predictions were close to 0.5 (0.53, 0.51, 0.57). The model was unsure, but it still had to pick a side.
- The training data is long, formal professional reviews, so short casual sentences like my tests are a domain mismatch.

**How I'd improve it:**
1. Use sentence embeddings (Day 4) or a transformer, which capture context, negation and tone.
2. Train on more diverse data, including short and informal text.
3. Add features that handle negation, such as tagging words after "not".

**Key takeaway:** A model's accuracy score only tells you how it does on data that looks like its training set. Writing my own test cases showed what it doesn't understand.